# 时间对齐与重采样

学习目标：汇总不规则时间观测，区分时间网格与聚合结果，并按分组、方向和容差关联事件与测量值。

前置知识：时间索引、时区、时间差、表格连接、聚合与缺失值。

运行环境：Python 3.12、pandas 3.0。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例数据均在单元内构造，后续单元沿用已导入的 pd。时间均表示同一地点、不跨夏令时变化的本地时间，使用无时区时间戳；数据值的单位为 °C，另有说明的例子除外。

## 1 汇总不规则观测

传感器并不总按固定间隔上报。要查看每 5 分钟的平均温度，可以先把观测时间作为 DatetimeIndex，再用 resample 划分时间桶并求均值。

下面明确使用左闭右开区间，结果以左边界标记。10:00 这一桶包括 10:00 和 10:02，10:05 这一桶包括 10:05 和 10:09。mean 计算非缺失值的均值，count 记录非缺失观测数。

In [1]:
import pandas as pd

times = pd.to_datetime([
    "2026-01-01 10:00", "2026-01-01 10:02",
    "2026-01-01 10:05", "2026-01-01 10:09",
], format="%Y-%m-%d %H:%M")
readings = pd.DataFrame({"temperature_c": [10, 14, 20, 28]}, index=times)
readings.index.name = "time"
buckets = readings.resample("5min", closed="left", label="left")
summary = buckets.agg(["mean", "count"])
print(readings)
print(summary)  # 10:00 为均值 12、计数 2；10:05 为均值 24、计数 2。
print(summary.shape, summary.index.dtype)  # (2, 2)，本例为 datetime64[us]。
print(summary.dtypes)  # mean 为 float64，count 为 int64；两级列名标明统计项。

                     temperature_c
time                              
2026-01-01 10:00:00             10
2026-01-01 10:02:00             14
2026-01-01 10:05:00             20
2026-01-01 10:09:00             28
                    temperature_c      
                             mean count
time                                   
2026-01-01 10:00:00          12.0     2
2026-01-01 10:05:00          24.0     2
(2, 2) datetime64[us]
temperature_c  mean     float64
               count      int64
dtype: object


## 2 时间网格与聚合

### 2.1 asfreq 不求均值

对这里的 DatetimeIndex，asfreq 从首个时间到最后一个时间按指定频率建立网格，仅保留落在网格时间上的原值；新时间没有对应观测时出现缺失值。它不把区间内多条观测合成一个值。

继续使用 readings。两个方法都得到两行，但 asfreq 的 10:00 是这一时刻的原值，resample 的 10:00 是整个桶的均值。网格不一定保留原来的最后一条观测。

In [2]:
grid = readings.asfreq("5min")
averages = readings.resample("5min", closed="left", label="left").mean()
print(grid)  # 10:00 为 10，10:05 为 20；10:02 和 10:09 不在网格上。
print(averages)  # 相同时间标签下为 12.0、24.0。
print(grid.shape, averages.shape)  # 均为 (2, 1)，含义不同。

                     temperature_c
time                              
2026-01-01 10:00:00             10
2026-01-01 10:05:00             20
                     temperature_c
time                              
2026-01-01 10:00:00           12.0
2026-01-01 10:05:00           24.0
(2, 1) (2, 1)


### 2.2 上采样与填充方向

把每 2 分钟一次的序列改成每分钟一次，是上采样（upsampling）；把观测汇总到更稀疏的时间桶，是降采样（downsampling）。增加网格点不会产生新的测量事实。

resample 后调用 asfreq 可查看新网格上的缺口。ffill 将前一条观测沿时间向后传播，bfill 使用后一条观测补前面的缺口。下面 10:01 的 bfill 值来自 10:02；若任务只能使用当时已知的数据，就不能这样填。

In [3]:
regular = pd.Series(
    [10, 20, 30],
    index=pd.date_range("2026-01-01 10:00", periods=3, freq="2min"),
    name="temperature_c",
)
filled = pd.DataFrame({
    "original_grid": regular.resample("1min").asfreq(),
    "ffill": regular.resample("1min").ffill(),
    "bfill": regular.resample("1min").bfill(),
})
print(filled)  # 新增 10:01、10:03；ffill 分别为 10、20，bfill 为 20、30。
print(filled.shape)  # (5, 3)。

                     original_grid  ffill  bfill
2026-01-01 10:00:00           10.0     10     10
2026-01-01 10:01:00            NaN     10     20
2026-01-01 10:02:00           20.0     20     20
2026-01-01 10:03:00            NaN     20     30
2026-01-01 10:04:00           30.0     30     30
(5, 3)


### 2.3 原有缺失与新增缺失

asfreq 的 method 或重采样后的 ffill、bfill 用于补新网格位置，不会改写原观测中的缺失。原位置是 NaN 时，传播到相邻新位置的也可能是 NaN。

先建立网格、再对普通 Series 调用 ffill，则会处理结果中所有可前向填充的缺失，包括原有缺失。是否允许补原始缺测，需要另行决定。

In [4]:
missing = pd.Series(
    [10.0, None, 30.0],
    index=pd.date_range("2026-01-01 10:00", periods=3, freq="2min"),
)
comparison = pd.DataFrame({
    "asfreq_ffill": missing.asfreq("1min", method="ffill"),
    "resample_ffill": missing.resample("1min").ffill(),
    "resample_bfill": missing.resample("1min").bfill(),
    "fill_all": missing.asfreq("1min").ffill(),
})
print(comparison)
# 前两列相同：10:02、10:03 都缺失；bfill 在 10:01、10:02 缺失。
# fill_all 将 10:01、10:02、10:03 都填为 10.0；这改变了原始缺测。

                     asfreq_ffill  resample_ffill  resample_bfill  fill_all
2026-01-01 10:00:00          10.0            10.0            10.0      10.0
2026-01-01 10:01:00          10.0            10.0             NaN      10.0
2026-01-01 10:02:00           NaN             NaN             NaN      10.0
2026-01-01 10:03:00           NaN             NaN            30.0      10.0
2026-01-01 10:04:00          30.0            30.0            30.0      30.0


填充还可以限制连续新增位置的数量。limit 按位置计数，不是按实际经过的时间计数；固定网格下才可以换算成相应时长。

In [5]:
sparse = pd.Series(
    [10, 30],
    index=pd.to_datetime(["2026-01-01 10:00", "2026-01-01 10:04"]),
)
limited = sparse.resample("1min").ffill(limit=1)
print(limited)  # 只补 10:01；10:02、10:03 仍为 NaN，10:04 保留 30。

2026-01-01 10:00:00    10.0
2026-01-01 10:01:00    10.0
2026-01-01 10:02:00     NaN
2026-01-01 10:03:00     NaN
2026-01-01 10:04:00    30.0
Freq: min, dtype: float64


## 3 桶边界与结果标签

closed 决定区间哪一端包含边界值；label 决定结果用哪一端命名。只改 label 不会把观测移到另一桶。默认值随频率种类而异，需要固定口径时显式指定。

下面的数值表示事件数量。左闭区间 [10:00, 10:05) 包含 10:00、10:04；右闭区间 (10:00, 10:05] 包含 10:04、10:05。左端的方括号表示包含，圆括号表示不包含。

![重采样桶边界与结果标签：改变 label 保持成员不变，改变 closed 则改变边界归属](image/16-resample-boundaries.png)

示意图突出同一对边界之间的桶。前两行只更换标签，总和保持为 3；第三行改为右闭后，10:00 被排除、10:05 被纳入，总和变为 6。

In [6]:
counts = pd.Series(
    [1, 2, 4, 8, 16],
    index=pd.to_datetime([
        "2026-01-01 10:00", "2026-01-01 10:04", "2026-01-01 10:05",
        "2026-01-01 10:09", "2026-01-01 10:10",
    ]),
    name="count",
)
left = counts.resample("5min", closed="left", label="left").sum()
relabeled = counts.resample("5min", closed="left", label="right").sum()
right = counts.resample("5min", closed="right", label="right").sum()
print(left)  # 标签 10:00、10:05、10:10，值 3、12、16。
print(relabeled)  # 标签 10:05、10:10、10:15，值仍为 3、12、16。
print(right)  # 标签 10:00、10:05、10:10，值 1、6、24。

2026-01-01 10:00:00     3
2026-01-01 10:05:00    12
2026-01-01 10:10:00    16
Freq: 5min, Name: count, dtype: int64
2026-01-01 10:05:00     3
2026-01-01 10:10:00    12
2026-01-01 10:15:00    16
Freq: 5min, Name: count, dtype: int64
2026-01-01 10:00:00     1
2026-01-01 10:05:00     6
2026-01-01 10:10:00    24
Freq: 5min, Name: count, dtype: int64


## 4 移动与差分

### 4.1 按行移动

shift(1) 在索引不变的情况下把值向后移一行，首行补缺失；diff() 计算当前值减前一行的值。它们默认按行位置工作，遇到不规则采样时，“前一行”不等于“前一分钟”。

下面相邻时间分别相差 2 分钟和 5 分钟，差分只表示温度变化，单位仍为 °C，不是每分钟的变化率。

In [7]:
uneven = pd.Series(
    [10, 15, 20],
    index=pd.to_datetime([
        "2026-01-01 10:00", "2026-01-01 10:02", "2026-01-01 10:07",
    ]),
    name="temperature_c",
)
changes = pd.DataFrame({"value": uneven, "previous": uneven.shift(1), "change": uneven.diff()})
print(changes)  # previous 为 NaN、10、15；change 为 NaN、5、5。
print(changes.dtypes)  # 原值 int64；引入缺失的两列为 float64。

                     value  previous  change
2026-01-01 10:00:00     10       NaN     NaN
2026-01-01 10:02:00     15      10.0     5.0
2026-01-01 10:07:00     20      15.0     5.0
value         int64
previous    float64
change      float64
dtype: object


### 4.2 移动时间标签

shift 指定 freq 后移动时间索引，值保持原来的顺序与对应关系。下面给所有标签加 1 分钟，并没有按分钟重新采样。

diff 没有 freq 参数。要比较某个固定时间间隔的值，须先明确时间对齐方式，不能把不规则序列的 diff(1) 当成固定时长差分。

In [8]:
moved = uneven.shift(1, freq="1min")
print(moved)  # 标签变为 10:01、10:03、10:08，值仍为 10、15、20。
print(moved.index.dtype, moved.dtype)  # datetime64[us]，int64。
print(uneven.index.equals(moved.index))  # False；原 uneven 的标签没有改变。

2026-01-01 10:01:00    10
2026-01-01 10:03:00    15
2026-01-01 10:08:00    20
Name: temperature_c, dtype: int64
datetime64[us] int64
False


## 5 按时间匹配事件

### 5.1 找到最近一次历史测量

事件发生时间与测量时间通常不相等。merge_asof 为每一条左表事件寻找一个满足方向条件的右表记录，未匹配时右表字段缺失。

两张表都须按时间键全局递增排序。backward 选择不晚于事件时间的最后一条测量；默认允许时间相等。使用不同列名保留两侧时间，便于检查匹配间隔。下面假定测量在其时间戳所示时刻即可获得。

In [9]:
events = pd.DataFrame({"event_time": pd.to_datetime([
    "2026-01-01 10:01", "2026-01-01 10:05", "2026-01-01 10:09",
])})
measurements = pd.DataFrame({
    "measure_time": pd.to_datetime([
        "2026-01-01 10:00", "2026-01-01 10:04", "2026-01-01 10:10",
    ]),
    "temperature_c": [10, 40, 100],
})
matched = pd.merge_asof(
    events, measurements, left_on="event_time", right_on="measure_time",
    direction="backward",
)
matched["age"] = matched["event_time"] - matched["measure_time"]
print(matched)  # 三行温度为 10、40、40；测量分别早 1、1、5 分钟。
print(matched.shape)  # (3, 4)，每条事件一行。
print(matched.dtypes)  # 两个时间列为 datetime64[us]，age 为 timedelta64[us]。

           event_time        measure_time  temperature_c             age
0 2026-01-01 10:01:00 2026-01-01 10:00:00             10 0 days 00:01:00
1 2026-01-01 10:05:00 2026-01-01 10:04:00             40 0 days 00:01:00
2 2026-01-01 10:09:00 2026-01-01 10:04:00             40 0 days 00:05:00
(3, 4)
event_time        datetime64[us]
measure_time      datetime64[us]
temperature_c              int64
age              timedelta64[us]
dtype: object


### 5.2 匹配方向

direction 控制候选测量相对于事件的位置。

| 参数值 | 中文名称／含义 |
| --- | --- |
| backward | 向后查找：选择时间不晚于事件的最后一条记录 |
| forward | 向前查找：选择时间不早于事件的第一条记录 |
| nearest | 最近匹配：选择时间距离最小的记录 |

继续使用 measurements，检查 10:08 的事件。forward 与 nearest 都可能使用未来测量。如果只允许使用事件发生时已知的资料，应选择符合该约束的方向；nearest 不等于“最近的历史记录”。

![同一事件的三种 merge_asof 方向：backward 选择历史测量，forward 与 nearest 选择更近的未来测量](image/16-asof-directions.png)

示意图使用下面代码的输入。箭头从事件指向选中的测量；nearest 选择未来 2 分钟的记录，而非历史 4 分钟的记录。

In [10]:
one_event = pd.DataFrame({"event_time": pd.to_datetime(["2026-01-01 10:08"])})
for direction in ["backward", "forward", "nearest"]:
    result = pd.merge_asof(
        one_event, measurements, left_on="event_time", right_on="measure_time",
        direction=direction,
    )
    print(direction)
    print(result)
# backward 匹配 10:04 的 40；forward、nearest 均匹配未来 10:10 的 100。

backward
           event_time        measure_time  temperature_c
0 2026-01-01 10:08:00 2026-01-01 10:04:00             40
forward
           event_time        measure_time  temperature_c
0 2026-01-01 10:08:00 2026-01-01 10:10:00            100
nearest
           event_time        measure_time  temperature_c
0 2026-01-01 10:08:00 2026-01-01 10:10:00            100


### 5.3 容差与完全相等

tolerance 限制最大匹配距离，时间键使用兼容的 Timedelta。继续使用前面的三条 events：只接受 2 分钟以内的历史测量，最后一条事件就没有合格记录。

In [11]:
recent = pd.merge_asof(
    events, measurements, left_on="event_time", right_on="measure_time",
    direction="backward", tolerance=pd.Timedelta("2min"),
)
print(recent)  # 最后一行 measure_time 为 NaT、temperature_c 为 NaN。
print(recent["measure_time"].notna().tolist())  # [True, True, False]。
print(recent.shape, recent["temperature_c"].dtype)  # (3, 3)，float64。

           event_time        measure_time  temperature_c
0 2026-01-01 10:01:00 2026-01-01 10:00:00           10.0
1 2026-01-01 10:05:00 2026-01-01 10:04:00           40.0
2 2026-01-01 10:09:00                 NaT            NaN
[True, True, False]
(3, 3) float64


allow_exact_matches=False 排除时间相等的记录。对 backward，这把“不晚于”改为“严格早于”；适用于要求测量必须发生在事件之前的口径。

In [12]:
exact_event = pd.DataFrame({"event_time": pd.to_datetime(["2026-01-01 10:04"])})
for allow_exact in [True, False]:
    result = pd.merge_asof(
        exact_event, measurements, left_on="event_time", right_on="measure_time",
        direction="backward", allow_exact_matches=allow_exact,
    )
    print(allow_exact, result["measure_time"].iloc[0], result["temperature_c"].iloc[0])
# True 使用 10:04 的 40；False 改用 10:00 的 10。

True 2026-01-01 10:04:00 40
False 2026-01-01 10:00:00 10


## 6 分组、排序与时间键

### 6.1 只在同一设备内匹配

by 先限定分组键相等，再按时间寻找候选记录。即使有 by，两表的时间键仍须各自全局递增；不要求把相同设备排在一起。

下面设备 A、B 的记录按时间交错排列。若省略设备条件，A 的事件就可能错误地使用 B 的测量。

In [13]:
device_events = pd.DataFrame({
    "device": ["A", "B", "A"],
    "event_time": pd.to_datetime([
        "2026-01-01 10:03", "2026-01-01 10:04", "2026-01-01 10:06",
    ]),
})
device_values = pd.DataFrame({
    "device": ["A", "B", "A"],
    "measure_time": pd.to_datetime([
        "2026-01-01 10:00", "2026-01-01 10:02", "2026-01-01 10:05",
    ]),
    "temperature_c": [10, 20, 15],
})
grouped = pd.merge_asof(
    device_events, device_values, by="device",
    left_on="event_time", right_on="measure_time", direction="backward",
)
print(grouped)  # A、B、A 分别得到 10、20、15，匹配时间为 10:00、10:02、10:05。
print(grouped.shape, grouped["device"].dtype)  # (3, 4)，str。

  device          event_time        measure_time  temperature_c
0      A 2026-01-01 10:03:00 2026-01-01 10:00:00             10
1      B 2026-01-01 10:04:00 2026-01-01 10:02:00             20
2      A 2026-01-01 10:06:00 2026-01-01 10:05:00             15
(3, 4) str


先按设备、再按时间排序，只保证组内有序，不保证整个时间列有序。下面故意造成 10:03、10:06、10:04 的排列；修正时对时间键排序即可。

In [14]:
wrong_order = device_events.sort_values(["device", "event_time"])
print(wrong_order["event_time"].is_monotonic_increasing)  # False。
try:
    pd.merge_asof(
        wrong_order, device_values, by="device",
        left_on="event_time", right_on="measure_time", direction="backward",
    )
except ValueError as error:
    print(type(error).__name__, str(error))  # left keys must be sorted。
else:
    raise AssertionError("全局时间键乱序时应失败")

fixed_order = wrong_order.sort_values("event_time")
print(fixed_order["event_time"].is_monotonic_increasing)  # True。

False
ValueError left keys must be sorted
True


### 6.2 时间键不能缺失

merge_asof 的时间键不能含 NaT。排序不会修复缺失的时间；应根据任务约定补正时间，或单独保留这些记录并报告未参与匹配，而不是悄悄把它们当作正常事件。

这里沿用 events 和 measurements，只将第一条事件的时间设为空。

In [15]:
unknown_time = events.copy()
unknown_time.loc[0, "event_time"] = pd.NaT
try:
    pd.merge_asof(
        unknown_time.sort_values("event_time"), measurements,
        left_on="event_time", right_on="measure_time",
    )
except ValueError as error:
    print(type(error).__name__, str(error))  # Merge keys contain null values on left side。
else:
    raise AssertionError("缺失时间键应被拒绝")
print(unknown_time["event_time"].isna().sum())  # 1 条事件需要另行处理。

ValueError Merge keys contain null values on left side
1


### 6.3 类型与时间单位一致

两侧时间键须有相同 dtype，分组键也要类型匹配。时间键还涉及时间单位与时区，不能仅凭打印出来的日期相同就判断兼容。

下面原输入解析为微秒单位。把右表改为纳秒单位后，即使表示的时刻不变，merge_asof 仍会拒绝。dt.as_unit 用于显式转换单位；应先确认范围与精度要求，再统一两侧。本例没有微秒以下的小数，转回微秒不会丢失信息。

In [16]:
nanosecond_values = measurements.copy()
nanosecond_values["measure_time"] = nanosecond_values["measure_time"].dt.as_unit("ns")
print(events["event_time"].dtype, nanosecond_values["measure_time"].dtype)
# datetime64[us] 与 datetime64[ns]。
try:
    pd.merge_asof(
        events, nanosecond_values, left_on="event_time", right_on="measure_time",
    )
except pd.errors.MergeError as error:
    print(type(error).__name__, str(error))  # must be the same type。
else:
    raise AssertionError("不同时间单位的键应被拒绝")

nanosecond_values["measure_time"] = nanosecond_values["measure_time"].dt.as_unit("us")
print(nanosecond_values["measure_time"].equals(measurements["measure_time"]))  # True。

datetime64[us] datetime64[ns]
MergeError incompatible merge keys [0] dtype('<M8[us]') and dtype('<M8[ns]'), must be the same type
True


## 7 选学：调整时间桶起点

对分钟、小时等固定长度频率，origin 决定时间桶的参照点，offset 在该参照点上增加偏移。origin="start_day" 以首日午夜为参照，origin="start" 以首条观测为参照。这些起点调整不适用于月、季度等非固定长度频率；带时区索引的 origin 还须与索引时区一致。

下面数值为事件数量。同样是 5 分钟一桶，从 10:00 还是 10:02 开始会改变分组。

In [17]:
offset_counts = pd.Series(
    [2, 5, 8],
    index=pd.to_datetime([
        "2026-01-01 10:02", "2026-01-01 10:05", "2026-01-01 10:08",
    ]),
)
midnight_bins = offset_counts.resample(
    "5min", origin="start_day", closed="left", label="left",
).sum()
start_bins = offset_counts.resample(
    "5min", origin="start", closed="left", label="left",
).sum()
offset_bins = offset_counts.resample(
    "5min", origin="start_day", offset="2min", closed="left", label="left",
).sum()
print(midnight_bins)  # 10:00 为 2，10:05 为 13。
print(start_bins)  # 10:02 为 7，10:07 为 8。
print(offset_bins.equals(start_bins))  # True；本例偏移 2 分钟得到相同分桶。

2026-01-01 10:00:00     2
2026-01-01 10:05:00    13
Freq: 5min, dtype: int64
2026-01-01 10:02:00    7
2026-01-01 10:07:00    8
Freq: 5min, dtype: int64
True


## 8 选学：按有序键合并

merge_ordered 适合按有序键合并数据。默认 outer 保留两侧键的并集，fill_method="ffill" 可将前面的记录向后传播。它与 merge_asof 保留每条左表记录、为其寻找近邻的任务不同。

下面两台设备在不同时间上报。先保留两侧的全部观测时刻，再查看前向填充的效果；没有更早记录的位置仍为空。

In [18]:
sensor_a = pd.DataFrame({
    "time": pd.to_datetime(["2026-01-01 10:00", "2026-01-01 10:04"]),
    "a_c": [10, 40],
})
sensor_b = pd.DataFrame({
    "time": pd.to_datetime(["2026-01-01 10:02", "2026-01-01 10:04"]),
    "b_c": [20, 41],
})
ordered = pd.merge_ordered(sensor_a, sensor_b, on="time", how="outer")
carried = pd.merge_ordered(sensor_a, sensor_b, on="time", how="outer", fill_method="ffill")
print(ordered)  # 按时间排列三行；10:00 缺 b_c，10:02 缺 a_c。
print(carried)  # a_c 为 10、10、40；b_c 为 NaN、20、41。
print(carried.shape, carried["time"].dtype)  # (3, 3)，datetime64[us]。

                 time   a_c   b_c
0 2026-01-01 10:00:00  10.0   NaN
1 2026-01-01 10:02:00   NaN  20.0
2 2026-01-01 10:04:00  40.0  41.0
                 time  a_c   b_c
0 2026-01-01 10:00:00   10   NaN
1 2026-01-01 10:02:00   10  20.0
2 2026-01-01 10:04:00   40  41.0
(3, 3) datetime64[us]


## 本章小结

（1）asfreq 重建时间网格，resample 按时间桶组织数据并聚合。标签相同不代表数值含义相同。

（2）closed 控制边界值归属，label 控制结果标签；origin、offset 可调整固定长度时间桶的参照点。

（3）重采样填充区分新网格缺口与原有缺失。bfill 使用后面的记录；填充方法须符合资料可用时间的约束。

（4）shift 不带 freq 时移动值，带 freq 时移动时间标签；diff 默认比较相邻行，不代表固定时间间隔。

（5）merge_asof 要核对全局排序、键类型与非缺失条件，再明确分组、方向、容差和是否允许相等。保留两侧时间有助于发现跨组或未来匹配。

## 练习

（1）将下面的不规则温度观测汇总到 5 分钟桶，显式指定左闭、左标签，分别求均值与计数。再用 asfreq 建立 5 分钟网格，解释两个结果第一行为什么不同。

In [19]:
exercise_readings = pd.Series(
    [12, 18, 24],
    index=pd.to_datetime([
        "2026-01-01 09:00", "2026-01-01 09:03", "2026-01-01 09:05",
    ]),
    name="temperature_c",
)
# 在此汇总、重建网格并打印结果。
# 检查：两个桶标签为 09:00、09:05；均值为 15、24，计数为 2、1。
# 网格只保留对应时刻的原值；同时检查结果的形状和 dtype。

（2）先预测两种填充在每个时间上的值，再运行。随后说明哪一种操作修改了原有缺失，以及在原始缺测必须保留时应选择哪一种。

In [20]:
exercise_missing = pd.Series(
    [5.0, None, 9.0],
    index=pd.date_range("2026-01-01 09:00", periods=3, freq="2min"),
)
print(exercise_missing.asfreq("1min", method="ffill"))
print(exercise_missing.asfreq("1min").ffill())
# 先写逐行预测，再比较原有时间位置与新增位置的缺失状态。
# 在此解释原始缺测必须保留时的方法选择。

2026-01-01 09:00:00    5.0
2026-01-01 09:01:00    5.0
2026-01-01 09:02:00    NaN
2026-01-01 09:03:00    NaN
2026-01-01 09:04:00    9.0
Freq: min, dtype: float64
2026-01-01 09:00:00    5.0
2026-01-01 09:01:00    5.0
2026-01-01 09:02:00    5.0
2026-01-01 09:03:00    5.0
2026-01-01 09:04:00    9.0
Freq: min, dtype: float64


（3）事件只能使用同设备、严格早于事件、最多相隔 2 分钟的测量。选择 merge_asof 的参数并说明理由，保留两侧时间检查结果。若条件改成“事后分析，允许使用前后 2 分钟内最近的一次测量，也允许同时发生”，应修改哪些参数？给出改变后的结果。

In [21]:
exercise_events = pd.DataFrame({
    "device": ["A", "B", "A"],
    "event_time": pd.to_datetime([
        "2026-01-01 09:03", "2026-01-01 09:04", "2026-01-01 09:08",
    ]),
})
exercise_values = pd.DataFrame({
    "device": ["A", "B", "A", "A"],
    "measure_time": pd.to_datetime([
        "2026-01-01 09:02", "2026-01-01 09:04",
        "2026-01-01 09:05", "2026-01-01 09:09",
    ]),
    "temperature_c": [12, 20, 15, 19],
})
# 在此完成两种约束下的匹配，并在注释中解释参数选择。
# 检查原条件：三行只有第一行匹配到 12，后两行测量时间为 NaT。
# 检查新条件：三行分别为 12、20、19；最后一行用了未来记录。
# 检查分组一致、时间差满足限制，结果行数等于事件数。

（4）对下面相邻间隔不同的序列，分别执行 shift(1)、shift(1, freq="2min") 和 diff()。打印索引与值，说明哪个操作改变了时间标签，为什么 diff 的两个非缺失结果不能直接解释为每分钟变化量。

In [22]:
exercise_uneven = pd.Series(
    [10, 14, 18],
    index=pd.to_datetime([
        "2026-01-01 09:00", "2026-01-01 09:02", "2026-01-01 09:06",
    ]),
)
# 在此分别计算并打印三种结果，记录变化后的标签和 dtype。
# 检查：按频率移动后的标签为 09:02、09:04、09:08，值与原值一致。
# 差分为 NaN、4、4；解释两个 4 对应的时间间隔有何不同。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | 频率与填充：[DataFrame.asfreq](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.asfreq.html) 的 DatetimeIndex 网格、method 与原有缺失；[DataFrame.resample](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html) 的 closed、label、origin、offset 和 Examples；[Time series — Resampling](https://pandas.pydata.org/docs/user_guide/timeseries.html#resampling) 的 Basics、Upsampling 与桶调整；[Resampler.mean](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.Resampler.mean.html)、[count](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.Resampler.count.html) 的非缺失统计；[Resampler.ffill](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.Resampler.ffill.html)、[bfill](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.Resampler.bfill.html) 的新位置填充与 limit；[Series.ffill](https://pandas.pydata.org/docs/reference/api/pandas.Series.ffill.html) 的缺失传播。移动与匹配：[DataFrame.shift](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shift.html) 的 periods、freq；[DataFrame.diff](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.diff.html) 的相邻差分；[merge_asof](https://pandas.pydata.org/docs/reference/api/pandas.merge_asof.html) 的全局时间排序、by、direction、tolerance、allow_exact_matches；[merge_ordered](https://pandas.pydata.org/docs/reference/api/pandas.merge_ordered.html) 的 how 与 fill_method。时间输入：[to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) 的 format、utc、Returns；[date_range](https://pandas.pydata.org/docs/reference/api/pandas.date_range.html) 的 freq、periods、unit；[Series.dt.as_unit](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.as_unit.html) 的单位转换。 |
| GitHub 官方项目（版本化来源） | [v3.0.0 pandas/core/reshape/merge.py](https://github.com/pandas-dev/pandas/blob/v3.0.0/pandas/core/reshape/merge.py) 的 _AsOfMerge._maybe_require_matching_dtypes 与 _convert_values_for_libjoin：键类型一致、时间键非空及排序检查；同时核对了当前安装的 pandas 3.0.6 对应实现。 pandas v3.0.6 文档源码：[timeseries](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/timeseries.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |